# Euribor Rate Explorer

**Data source:** [euriborrates.com](https://euriborrates.com/) — an independent, non-commercial, non-profit information resource.

This notebook fetches Euribor fixings (1W, 1M, 3M, 6M, 12M) and displays the current state of our dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import pandas as pd
from datetime import date

from pricebook.data.euribor_loader import (
    fetch_date, fetch_year_all_tenors, load_local, save_to_csv,
    summary, attribution, TENORS, TENOR_LABELS,
)

print(attribution())

## 1. Fetch Today's Fixing

In [ ]:
# Fetch today's rates (all 5 tenors)
today = date.today()
fixing = fetch_date(today)

if fixing:
    print(f"Euribor fixings for {fixing.date}")
    print(f"{'Tenor':<12} {'Rate':>8}")
    print("-" * 22)
    for tenor in TENORS:
        rate = fixing.rates.get(tenor)
        if rate is not None:
            print(f"{TENOR_LABELS[tenor]:<12} {rate*100:>7.3f}%")
else:
    print(f"No fixing available for {today} (weekend or holiday?)")

## 2. Fetch a Sample Year (all 5 tenors)

In [ ]:
# Fetch 2024 with all 5 tenors (5 requests, ~10s)
fixings_2024 = fetch_year_all_tenors(2024, delay_between_tenors=1.5)
print(f"Fetched {len(fixings_2024)} business days for 2024")

# Convert to DataFrame
rows = []
for f in fixings_2024:
    row = {"date": f.date}
    for t in TENORS:
        row[t] = f.rates.get(t)
    rows.append(row)

df_2024 = pd.DataFrame(rows).set_index("date")
df_2024.columns = [TENOR_LABELS[t] for t in TENORS]
df_2024 = df_2024 * 100  # convert to percentage

print(f"\nFirst 5 days:")
df_2024.head()

In [ ]:
# Last 5 days
df_2024.tail()

## 3. Term Structure Snapshot

Current Euribor curve shape: how rates vary by maturity.

In [ ]:
import matplotlib.pyplot as plt

# Term structure: latest day in our 2024 data
latest = df_2024.iloc[-1]
tenor_months = [1/4, 1, 3, 6, 12]  # approximate months for each tenor

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tenor_months, latest.values, "o-", color="#1a5276", linewidth=2, markersize=8)
ax.set_xlabel("Tenor (months)")
ax.set_ylabel("Rate (%)")
ax.set_title(f"Euribor Term Structure — {df_2024.index[-1]}\nSource: euriborrates.com")
ax.set_xticks(tenor_months)
ax.set_xticklabels(["1W", "1M", "3M", "6M", "12M"])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Rate History (2024)

All 5 tenors through the year — observe the ECB easing cycle.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = ["#85c1e9", "#3498db", "#1a5276", "#e74c3c", "#c0392b"]
for i, col in enumerate(df_2024.columns):
    ax.plot(df_2024.index, df_2024[col], label=col, color=colors[i], linewidth=1.2)

ax.set_xlabel("Date")
ax.set_ylabel("Rate (%)")
ax.set_title("Euribor Rates — 2024\nSource: euriborrates.com")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Curve Slope (12M − 1W spread)

Positive = normal (upward sloping). Negative = inverted.

In [ ]:
slope = df_2024["12 Months"] - df_2024["1 Week"]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.fill_between(slope.index, slope.values, 0, 
                where=slope >= 0, color="#27ae60", alpha=0.4, label="Normal")
ax.fill_between(slope.index, slope.values, 0, 
                where=slope < 0, color="#e74c3c", alpha=0.4, label="Inverted")
ax.plot(slope.index, slope.values, color="#2c3e50", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("Spread (pp)")
ax.set_title("Euribor Slope: 12M − 1W (2024)\nSource: euriborrates.com")
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Statistics

In [ ]:
print("2024 Euribor Statistics (in %)")
print("=" * 55)
stats = df_2024.describe().T[["mean", "std", "min", "max"]]
stats.columns = ["Mean", "StdDev", "Min", "Max"]
print(stats.round(3).to_string())
print(f"\nBusiness days: {len(df_2024)}")
print(f"Date range: {df_2024.index[0]} to {df_2024.index[-1]}")
print(f"\nData source: euriborrates.com")

---

## Next Steps

1. **Full historical load** — fetch 1999–2026 (all 5 tenors, ~135 requests)
2. **Curve calibration** — build EUR deposit curve for each business day
3. **Daily cron** — automated daily update
4. **Analytics** — PCA, regime detection, ECB impact, carry backtest

*Data sourced from [euriborrates.com](https://euriborrates.com/)*